In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [17]:
#1 Import the loan data as a data frame and ensure that the data is loaded properly.
df = pd.read_csv("Loan_Train.csv")

df.head()
df.info

<bound method DataFrame.info of       Loan_ID  Gender Married Dependents     Education Self_Employed  \
0    LP001002    Male      No          0      Graduate            No   
1    LP001003    Male     Yes          1      Graduate            No   
2    LP001005    Male     Yes          0      Graduate           Yes   
3    LP001006    Male     Yes          0  Not Graduate            No   
4    LP001008    Male      No          0      Graduate            No   
..        ...     ...     ...        ...           ...           ...   
609  LP002978  Female      No          0      Graduate            No   
610  LP002979    Male     Yes         3+      Graduate            No   
611  LP002983    Male     Yes          1      Graduate            No   
612  LP002984    Male     Yes          2      Graduate            No   
613  LP002990  Female      No          0      Graduate           Yes   

     ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0               5849          

In [18]:
# 2 Prepare data for modeling

# Drop Loan_ID
df = df.drop(columns=['Loan_ID'])

# Drop rows with missing data
df = df.dropna()

# --- Split BEFORE dummy encoding ---
target = "Loan_Status"

# Separate features and target
X = df.drop(columns=[target])
y = df[target]

# Convert categorical features into dummy variables
categorical_cols = X.select_dtypes(exclude=['number']).columns
X = pd.get_dummies(X, columns=categorical_cols, drop_first=False)

In [19]:
# 3 Train/test split dataset
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [21]:
# 4 Create a pipeline with a min-max scaler and a KNN classifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier

# Create the pipeline
knn_pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('knn', KNeighborsClassifier())
])

In [23]:
# 5 Fit the default KNN pipeline
knn_pipeline.fit(X_train, y_train)

# Predict on the test set
y_pred = knn_pipeline.predict(X_test)

# Accuracy
from sklearn.metrics import accuracy_score
default_accuracy = accuracy_score(y_test, y_pred)

print("5. Default KNN Test Accuracy:", default_accuracy)

5. Default KNN Test Accuracy: 0.78125


In [24]:
# 6 Search space for KNN hyperparameter
param_grid = {
    'knn__n_neighbors': list(range(1, 11))
}

In [25]:
# 7 Grid search with 5-fold CV
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy'
)

grid.fit(X_train, y_train)

print("Best n_neighbors:", grid.best_params_)
print("Best CV Accuracy:", grid.best_score_)

Best n_neighbors: {'knn__n_neighbors': 9}
Best CV Accuracy: 0.7057416267942583


In [26]:
# 8 Find the accuracy of the grid search best model on the test set. Note: It is possible that this will not be an improvement over the default model, 
#   but likely it will be.
best_model = grid.best_estimator_

y_pred_best = best_model.predict(X_test)
best_accuracy = accuracy_score(y_test, y_pred_best)

print("8. Best Model Test Accuracy:", best_accuracy)

8. Best Model Test Accuracy: 0.75


In [29]:
# 9 Now, repeat steps 6 and 7 with the same pipeline, but expand your search space to include logistic regression and random forest models with the 
#   hyperparameter values in section 12.3 of the Machine Learning with Python Cookbook.

# Pipeline for multi-model search
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('model', KNeighborsClassifier())  
])

In [30]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Expanded search space using section 12.3 hyperparameters
search_space = [

    # Logistic Regression
    {
        'model': [LogisticRegression(max_iter=500, solver='liblinear')],
        'model__penalty': ['l1', 'l2'],
        'model__C': np.logspace(0, 4, 10)
    },

    # Random Forest
    {
        'model': [RandomForestClassifier()],
        'model__n_estimators': [10, 100, 1000],
        'model__max_features': [1, 2, 3]
    }
]

In [31]:
# Running grid search with 5‑fold CV
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=search_space,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', MinMaxScaler()),
                                       ('model', KNeighborsClassifier())]),
             n_jobs=-1,
             param_grid=[{'model': [LogisticRegression(C=2.7825594022071245,
                                                       max_iter=500,
                                                       solver='liblinear')],
                          'model__C': array([1.00000000e+00, 2.78255940e+00, 7.74263683e+00, 2.15443469e+01,
       5.99484250e+01, 1.66810054e+02, 4.64158883e+02, 1.29154967e+03,
       3.59381366e+03, 1.00000000e+04]),
                          'model__penalty': ['l1', 'l2']},
                         {'model': [RandomForestClassifier()],
                          'model__max_features': [1, 2, 3],
                          'model__n_estimators': [10, 100, 1000]}],
             scoring='accuracy')

In [34]:
# 10 Now, repeat steps 6 and 7 with the same pipeline, but expand your search space to include logistic regression and random forest models with the hyperparameter values in section 12.3
#    of the Machine Learning with Python Cookbook.

# Report the best model with hyperparameters
print("Best Model:", grid.best_estimator_)
print("Best Parameters:", grid.best_params_)
print("Best CV Accuracy:", grid.best_score_)

Best Model: Pipeline(steps=[('scaler', MinMaxScaler()),
                ('model',
                 LogisticRegression(C=2.7825594022071245, max_iter=500,
                                    solver='liblinear'))])
Best Parameters: {'model': LogisticRegression(C=2.7825594022071245, max_iter=500, solver='liblinear'), 'model__C': 2.7825594022071245, 'model__penalty': 'l2'}
Best CV Accuracy: 0.8100136705399864


In [33]:
# Evaluate the best model on the test set
from sklearn.metrics import accuracy_score

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)

print("Test Accuracy of Best Model:", test_accuracy)

Test Accuracy of Best Model: 0.8229166666666666
